In [ ]:
# Run first on Google Colab
!pip install qutip -q

# §5 Grover's Algorithm and Amplitude Estimation

**Course:** Introductory Quantum Computing — Summer School

## Learning objectives
- Implement Grover's algorithm and track the geometric rotation in the $\{|\alpha\rangle,|\beta\rangle\}$ plane
- Verify the optimal step count $k^* = \lfloor\pi/(4\theta)\rfloor$ experimentally
- Implement amplitude estimation via QPE on the Grover operator


In [ ]:
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
from scipy.linalg import expm

print(f"QuTiP {qt.__version__}")

# ── Helpers from Notebook 1 ───────────────────────────────────────────────────
def qft_matrix(n):
    N = 2**n
    omega = np.exp(2j * np.pi / N)
    F = np.array([[omega**(j*k) / np.sqrt(N) for k in range(N)] for j in range(N)])
    return qt.Qobj(F, dims=[[2]*n, [2]*n])

def apply_to_qubit(gate, n, qubit):
    ops = [qt.qeye(2)] * n; ops[qubit] = gate
    return qt.tensor(ops)

def controlled_u(U, n, ctrl, tgt):
    P0 = qt.basis(2,0)*qt.basis(2,0).dag()
    P1 = qt.basis(2,1)*qt.basis(2,1).dag()
    ops0 = [qt.qeye(2)]*n; ops0[ctrl] = P0
    ops1 = [qt.qeye(2)]*n; ops1[ctrl] = P1; ops1[tgt] = U
    return qt.tensor(ops0) + qt.tensor(ops1)

def swap_gate(n, i, j):
    N = 2**n; mat = np.zeros((N,N),dtype=complex)
    for k in range(N):
        bits = list(format(k,f'0{n}b')); bits[i],bits[j]=bits[j],bits[i]
        mat[int(''.join(bits),2),k]=1.0
    return qt.Qobj(mat, dims=[[2]*n,[2]*n])

def qft_circuit(n):
    U = qt.tensor([qt.qeye(2)]*n); H1 = qt.gates.hadamard_transform(1)
    for i in range(n):
        U = apply_to_qubit(H1,n,i)*U
        for j in range(i+1,n):
            m=j-i+1; Rm=qt.Qobj(np.diag([1.0,np.exp(2j*np.pi/2**m)]))
            U = controlled_u(Rm,n,j,i)*U
    for i in range(n//2): U = swap_gate(n,i,n-1-i)*U
    return U

# ── QPE (carried over from Notebook 3) ────────────────────────────────────────
def qpe(U_system, eigenstate, t_clock):
    """Run QPE; return the full clock⊗system output state."""
    sys_n   = int(np.round(np.log2(U_system.shape[0])))
    I_sys   = qt.qeye(U_system.shape[0]); I_sys.dims = [[2]*sys_n, [2]*sys_n]
    total_n = t_clock + sys_n

    psi = qt.tensor(qt.tensor([qt.basis(2,0)]*t_clock), eigenstate)
    H_full = qt.tensor(qt.gates.hadamard_transform(t_clock), I_sys)
    H_full.dims = [[2]*total_n, [2]*total_n]
    psi = H_full * psi

    for j in range(t_clock):
        U_pow = U_system.copy()
        for _ in range(t_clock - 1 - j):
            U_pow = U_pow * U_pow
        ops0 = [qt.qeye(2)]*t_clock; ops0[j] = qt.basis(2,0)*qt.basis(2,0).dag()
        ops1 = [qt.qeye(2)]*t_clock; ops1[j] = qt.basis(2,1)*qt.basis(2,1).dag()
        gate = qt.tensor(qt.tensor(ops0), I_sys) + qt.tensor(qt.tensor(ops1), U_pow)
        gate.dims = [[2]*total_n, [2]*total_n]
        psi = gate * psi

    QFT_full = qt.tensor(qft_circuit(t_clock).dag(), I_sys)
    QFT_full.dims = [[2]*total_n, [2]*total_n]
    return QFT_full * psi

def measure_clock(psi_out, t_clock, sys_n):
    """Marginal probability distribution over the 2^t clock outcomes."""
    N_sys, N_clock = 2**sys_n, 2**t_clock
    amps = psi_out.full().flatten()
    return np.array([sum(abs(amps[k*N_sys + s])**2 for s in range(N_sys))
                     for k in range(N_clock)])

In [ ]:
#@title 🔒 Solution vault — enter the key from class to unlock  {display-mode: "form"}
# Solutions are obfuscated. Use the unlock key your instructor hands out in
# class, then run the relevant exercise's cell, e.g. reveal_solution("1.1", key="...").
import base64, zlib, hashlib

_VAULT = {
    "5.1": "LR1bH0/rjmCuX7UkJZPEGDXBIopkT25B5ZM5L3d5stYylkarx5XJ9SQ4f0ZmFf5qWWNzYzQv0zC56xjf39sNSqgy1CqfFfOflGUSR6F0M3Loqui0WI7K6rLHYSidqfSeamVJoMsSwM8C3RxPUiP8oahjINFfGwev3of7ESzCLVumaF+9b1Re9SQnoVwBrL80KOqESYPyGWE19bGyA8dt3ASyW0Ze3vOJRQ1LCRDHnhhk+42VF+UgDh5jsXiRlT6mncG1N5HjkE/nACG2hCi8heG8fCwBnTg8C9jiPryguGgrpiGaUWO6GV26KHEoJ6vC2IPf2DzNM1Z3syviBYcnOvtVGzlM54bKG/iU2/0+oeX93Ok7tXtp2Nrc1+oE1mtBRJsd9jiivsB29vWVsb+YkZbsg5Q+cy+mha0DYqlQfjDwAuHbC8nXKGUhGzOaTd61KGdZkVDYXgP0Twwv+1mXn5tQL2Fc30HFdxg4S8g2GLtEKjg7VoXDduoLLMns/kPrW3NbbCSNMzA2c13BTdIKLvIztqxADKrk+NXH9SIPYlX4EqrU0TVK0tQOjEvgbkAnT3aiIaEc0QNzcXibZEGlzxMqPZtETcVPoAldv1wrt1tiIlvI7cdANHFORtFs3KjPqfAL7fCpPjyVJld9HBOxE2D7t86NihmVS4qJ8WP2trBC1zB9vpHeHLF6pAmUFaCLCtg05ZwBPopP0DhsqxfoIB4tphs21HIkTaUl2JcwJL6YY6y1XczZkpWUe9s4os6p9BWGQfwRVcQv16/+0dF/OAy39qUkK+tVHWx3CuqtlPnmfawDk9UH+lXHbsR+J3a9kKBvfabWUBIEzJ51Wtnggx7WUm+9IyIR3x/aKIkOkB3Uyt7wNELgIhZdNzB8owJwJSp34P46X5g10RsMlN/wzbKzkM66n5dp7RAf+LlaQsBR9rJjG3cjA/sJrXxrbmKGQiP6wdBja3Pm5whnnLLNyVvd5zWQfCiZHBYHaiAcpi1kcRH4tA9wJf7/HDYE3QHUK1Ch8s9zFQSh1i4qdv+TDitG2D2UAwbSQ+qBxJgSwWqsE+iyAdJVI9BW0grNTzCLy0jf7xF4lGkEgZAbPCOEV05jYy5T13nB9XLPllqO5CU6H5nWSHg7sumEFPwInU1NKmjIid7ACNlTPofKGveNdu6/OO3ZtrMNDwbK5tOauSvV5m4DM/YEe1izTVYP2pZ4E6DfL87Ph6sq6eR3Zr15lrXA2HRYiWCUyLJHO8CoF4B4RR9D+OmGUjG6lJ1n3H3QtYBHFEaZOs+bVN2veXSZjcL+yiknVPoYGjbk5uQjtOsmi+d/JKrf+egQ1h+l1K+HwhU02QpGrNJHSmxZ1X+tSaL86G0YshRVasCEl4NChrFYn1Q1T69p7BA2FZLLlyopI+R+fzcwR828re89p/TX21VFOLko4dPX6n/iBTm/VRlqGSyf8yYmHkjR4QJUyb7eSebsJuAO8IJcMtfkSGm+o8zlHWXhgikhmwAorfgog2KVr0Tp2P6W7ELlb+jrS+4KN/i67gCxzJyQPj/1fueociAfO4EvTfJV/SHHMMgN3IS/hWs/LU/Z8y5YS4cyMhbWXHH8A9NzvN15L3mvOfIxBmjXNB8aqG9JqjqtLGXTw71mWphmRM0eRslhZMyuCkUylcX9EB05oYtmbuwqn7SmAUATWw5tRXIYlRBEV15vYm3TmD7NyppEGJabDbDe8blmPqOohu7JV4Nk9s7Cg8zl6cZn+GnhWQkyvE4FmGckxJwS+AKzM9/2BPOwdNaG/74akwuyXesyZX/h/TBfSlZzjDYuRRnlYkypy79C5Uy/HnhDbukOFyqL2kaxkyk4USVPvYCpC7Pv5quPtFdBRqvT/yGdxR4ph6WdO2xmLtke5RZlHCAp1UcYlg42xRW/bNCBHM86KthvtSKMX9IILHn+ZIcHsZUbHRYfhCd9Py1Th0YrDBAHJ74z+jS/yecTQJitjruCdhrmG9Y2eeU0kL43v7FWCY5WuL6YPf7Jl4eGqzip6C8+05HOkKdVxEDJiL3dmQJa/PBzzzLbgT+W",
    "5.2": "LR1bH0/rXmQyX7VSOdDAGLuD41Shk6+B8rE9MZVigsluq0k3spa7RcwnpHF+eKAiYkfmf1FK72BFpTJj1Mb1KUj519RAyCY/SVn1eLcwTgYgCjBSSc23jH6rKDqxtkU9w+tp6iXcKoNCuoQHi4U3bTuknUB0aRwb1izj8ZuBPoFplzln6TkHK/6lrNbeE6uqh9XN60lg3zFNuxMm80/1iUtY29CdHopez2HiDMP8Dyfw6CERodBReGz2PnbmQ3C2fIC6jse1M91UbWk0FkrQFr4R/DXFYZcnJDGyg2bYGOrq0MSYYVH7GXXJCT1RapA9gNRvQyavt2xhtO6juxp5Kl/2Gb5vRCl1VTWe011R2oM4hMs3KYoyP4DRt+XLEJQVZe9oOmFMXwZA7JA3z1VP49BzOYDkeNNg2y1Ogpe3Jaq4RUHxVqhPnlgelfianbmYa+fasR0/NvUhYeBwC4I/l/dFHanKz2HfDeqXq4aG5g/snD2BJnxwEzlMF9Q5X8rL4lw2aNPYxt43/Pa/9wgoNkYLtIP/dsgl/rfFZ4cVhCgaN5wIwzruRw0s/MUAMpHMBhILDIRJ3lnQvgRbHRq06emdOtavAfzCQiHRo6OFuFD4o5ck6DOW7C1X3nthjvTrNyRTBolS9ul0weAUscJ93lOGvSZ3Ck2OmzkQFm5iUy42v9iHYR8HIDqytYgBZHns8yeOX+PAEmfaR4vOl35Q+JKeBRyRaRqXG9Vuyxp7ZvN2AqBLKDG9v54gHyvuXU47P1j7lGH5k2xcbp76tTfw2k0cddMxYqi5uKnV6VXYWU28S408WVueruve0n22SPpR8pDKXs9KGDlrWjBLFSRVT4scJAE1N+ponYII8Q==",
}

def reveal_solution(exercise, key=""):
    """Print the solution for `exercise` (e.g. "1.1"), given the class unlock key."""
    blob = _VAULT.get(str(exercise))
    if blob is None:
        print(f"No solution stored for exercise {exercise!r}."); return
    kb = key.strip().encode()
    if not kb:
        print("Enter the key your instructor gave you, e.g. "
              f'reveal_solution("{exercise}", key="...").'); return
    data = base64.b64decode(blob.encode())
    out = bytes(b ^ kb[i % len(kb)] for i, b in enumerate(data))
    if out[:4] != hashlib.sha256(kb).digest()[:4]:
        print("✗ Wrong key — check the key your instructor gave you for THIS notebook."); return
    try:
        print(zlib.decompress(out[4:]).decode("utf-8"))
    except Exception:
        print("✗ Wrong key — check the key your instructor gave you for THIS notebook.")


---
## Part 1: Grover's Algorithm

### 5.1 Setup and geometric picture

**Problem:** Search $\{0,1\}^n$ for $M$ marked elements using oracle $O_f|x\rangle = (-1)^{f(x)}|x\rangle$.

**Geometric picture** (Figure 5.1): Work in the 2-dimensional plane spanned by
$$|\alpha\rangle = \frac{1}{\sqrt{N-M}}\sum_{f(x)=0}|x\rangle \quad\text{(unmarked)}\qquad
  |\beta\rangle = \frac{1}{\sqrt{M}}\sum_{f(x)=1}|x\rangle \quad\text{(marked)}$$

The uniform superposition $|s\rangle = \mathsf{H}^{\otimes n}|0^n\rangle$ starts at angle $\theta$ from $|\alpha\rangle$, where $\sin\theta = \sqrt{M/N}$.

**Grover operator:**
$$G = D \cdot O_f, \qquad D = 2|s\rangle\langle s| - \mathbb{I} \quad\text{(diffusion = reflection about }|s\rangle)$$

Each application of $G$ **rotates by $2\theta$** towards $|\beta\rangle$.

**Theorem (Grover):** After $k^* = \lfloor\pi/(4\theta)\rfloor$ steps, measuring yields a marked element with probability $\geq 1 - M/N$.
For $M=1$: $k^* \approx \frac{\pi}{4}\sqrt{N}$ — quadratic speedup over classical $O(N)$.


In [ ]:
def make_oracle(n, marked):
    """
    Problem instance: the phase oracle  O_f |x> = (-1)^{f(x)} |x>.

    This is the ONLY object that depends on `marked` -- it encodes the
    search problem. The solver (run_grover) never sees `marked`, only O_f.
    """
    N = 2**n
    diag = np.ones(N)
    for m in marked:
        diag[m] = -1.0
    return qt.Qobj(np.diag(diag), dims=[[2]*n, [2]*n])


def diffusion(n):
    """
    Diffusion D = 2|s><s| - I and the uniform superposition |s>.
    Independent of the marked set: built from |s> = H^{\u2297n}|0^n> alone.
    """
    s = qt.gates.hadamard_transform(n) * qt.tensor([qt.basis(2,0)]*n)
    D = 2 * s * s.dag() - qt.tensor([qt.qeye(2)]*n)
    return D, s


def run_grover(O_f, M, n_steps=None):
    """
    Grover solver. Receives only the black-box oracle O_f and the count M
    of marked elements -- never their positions. If n_steps is None, use the
    optimal k* = floor(pi/(4*theta)) with theta = arcsin(sqrt(M/N)).
    Returns (states, k_opt, theta).
    """
    n = len(O_f.dims[0])
    N = 2**n
    theta = np.arcsin(np.sqrt(M/N))
    k_opt = int(np.floor(np.pi / (4*theta)))
    if n_steps is None:
        n_steps = k_opt

    D, s = diffusion(n)
    G = D * O_f    # Grover operator

    psi = s.copy()
    states = [psi]
    for _ in range(n_steps):
        psi = G * psi
        states.append(psi.copy())

    return states, k_opt, theta

print("Grover helpers defined: make_oracle (problem)  +  diffusion, run_grover (solver).")

In [ ]:
# ── Grover on n=3 qubits, 1 marked element ────────────────────────────────────
n, marked = 3, [5]   # mark |101> = index 5
N, M = 2**n, len(marked)

O_f = make_oracle(n, marked)                  # problem instance (sees `marked`)
states, k_opt, theta = run_grover(O_f, M)     # solver: only the oracle + count M

print(f"n={n}, N={N}, M={M}")
print(f"θ = arcsin(√(M/N)) = {np.degrees(theta):.2f}°")
print(f"Optimal steps k* = ⌊π/(4θ)⌋ = {k_opt}")

# Probability of measuring marked element at each step
probs_marked = [abs(state.full().flatten()[marked[0]])**2 for state in states]
print(f"\nP(marked) at each step:")
for k, p in enumerate(probs_marked):
    bar = '█' * int(p*30)
    print(f"  k={k}: {p:.4f}  {bar}")

# ── Plot probability vs iteration ─────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(len(probs_marked)), probs_marked, 'o-')
ax1.axvline(k_opt, color='red', linestyle='--', label=f'k*={k_opt}')
ax1.set_xlabel('Grover iterations k')
ax1.set_ylabel('P(marked element)')
ax1.set_title(f'Grover convergence (n={n}, M={M}, marked={marked})')
ax1.legend()

# ── Bar chart of state at step k* ─────────────────────────────────────────────
amps = states[k_opt].full().flatten()
ax2.bar(range(N), np.abs(amps)**2)
ax2.set_xticks(range(N)); ax2.set_xticklabels([f'|{k:03b}⟩' for k in range(N)], rotation=45)
ax2.set_ylabel('probability'); ax2.set_title(f'State after k*={k_opt} steps')
ax2.axvline(marked[0]-0.5, color='red', alpha=0.3); ax2.axvline(marked[0]+0.5, color='red', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ── Track rotation in {|alpha>, |beta>} plane ─────────────────────────────────
n, marked = 4, [3]   # n=4, 1 marked element
N = 2**n

O_f = make_oracle(n, marked)

D, s = diffusion(n)

# Basis vectors of the Grover subspace
alpha_vec = np.zeros(N);
for x in range(N):
    if x not in marked: alpha_vec[x] = 1/np.sqrt(N - len(marked))
beta_vec = np.zeros(N)
for m in marked: beta_vec[m] = 1/np.sqrt(len(marked))

alpha = qt.Qobj(alpha_vec, dims=[[2]*n,[1]*n])
beta  = qt.Qobj(beta_vec,  dims=[[2]*n,[1]*n])

theta = np.arcsin(np.sqrt(len(marked)/N))
k_max = int(2 * np.pi / (2*theta)) + 2  # a few full rotations

G = D * O_f
psi = s.copy()
alpha_coords = [complex(alpha.dag()*psi).real]
beta_coords  = [complex(beta.dag()*psi).real]

for _ in range(k_max):
    psi = G * psi
    alpha_coords.append(complex(alpha.dag()*psi).real)
    beta_coords.append(complex(beta.dag()*psi).real)

fig, ax = plt.subplots(figsize=(6, 6))
theta_vals = np.linspace(0, 2*np.pi, 200)
ax.plot(np.cos(theta_vals), np.sin(theta_vals), 'gray', alpha=0.3)
ax.plot(alpha_coords, beta_coords, 'o-', markersize=6)
for i, (a, b) in enumerate(zip(alpha_coords, beta_coords)):
    ax.annotate(f'k={i}', (a, b), textcoords='offset points', xytext=(5,5), fontsize=8)
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.set_xlabel('⟨α|ψ_k⟩'); ax.set_ylabel('⟨β|ψ_k⟩')
ax.set_title(f'Grover rotation in {{|α⟩,|β⟩}} plane (n={n}, M=1)')
ax.set_aspect('equal'); plt.tight_layout(); plt.show()


### Exercise 5.1 — Grover operator as a rotation (Quantum Counting)

**(a)** Verify numerically that the Grover operator $G$ restricted to $\mathrm{span}\{|\alpha\rangle,|\beta\rangle\}$
acts as a rotation by $2\theta$, i.e., the $2\times 2$ matrix of $G$ in the $\{|\alpha\rangle,|\beta\rangle\}$ basis equals
$$G|_{V} = \begin{pmatrix}\cos 2\theta & -\sin 2\theta \\ \sin 2\theta & \cos 2\theta\end{pmatrix}$$

**(b)** Implement **quantum counting**: run QPE on $G$ to estimate $M$ from the eigenphase $\theta \approx \arcsin(\sqrt{M/N})$.
Use $n=4$ qubits with $M=2$ marked elements and $t=5$ clock qubits.

*Hint for (b):* The eigenvalues of $G$ (in the Grover subspace) are $e^{\pm 2i\theta}$, so QPE gives $\pm\theta/\pi$.


In [ ]:
# YOUR CODE HERE

n_ex, marked_ex = 4, [2, 7]   # M=2 marked elements, N=16

# (a) Build G, compute 2x2 matrix in {alpha, beta} basis
# O_f = make_oracle(n_ex, marked_ex)
# D, s_ex = diffusion(n_ex)
# G_ex = D * O_f
# G_ab = np.array([[...]])   # 2x2 matrix: rows/cols = alpha, beta

# theta_ex = np.arcsin(np.sqrt(len(marked_ex)/2**n_ex))
# R2theta = np.array([[np.cos(2*theta_ex), -np.sin(2*theta_ex)],
#                     [np.sin(2*theta_ex),  np.cos(2*theta_ex)]])

# (b) Quantum counting via QPE on G_ex with t=5 clock qubits.
# The eigenvalues of G are exp(±2i*theta), so QPE outputs phi ~ ±theta/pi.
# Recover M from: M = N * sin^2(pi * phi_estimated)
#
# Hint: define a qpe() function (or copy from Notebook 3).
# Use one of G's eigenstates in the Grover subspace as the system register.
# The eigenstate for eigenvalue exp(+2i*theta) is (|alpha> + i|beta>) / sqrt(2).

In [ ]:
#@title 🔒 Exercise 5.1 (locked)  {display-mode: "form"}
reveal_solution("5.1", key="PASTE-KEY-HERE")


### 5.2 Amplitude Estimation

**From quantum counting to amplitude estimation.** Quantum counting (§5.1) ran QPE on the Grover operator $G$ to read its eigenphase $2\theta$ and recover $\sin^2\theta = M/N$ — the probability that measuring $|s\rangle = H^{\otimes n}|0^n\rangle$ yields a marked element. Nothing there used the *uniform* state preparation: replacing $H^{\otimes n}$ by an arbitrary state-preparation circuit $A$ with
$$A|0\rangle = \sqrt{a}\,|\psi_1\rangle|1\rangle + \sqrt{1-a}\,|\psi_0\rangle|0\rangle$$
and marking the flag qubit $|1\rangle$ turns counting into a general primitive — **amplitude estimation** — that reads off the probability $a$ of *any* quantum subroutine. Counting is the special case $A = H^{\otimes n}$, $a = M/N$.

**Theorem (Amplitude Estimation):** QPE on $G_A$ with $t$ clock qubits outputs $\tilde{a}$ with
$$|\tilde{a}-a|\leq \frac{2\pi\sqrt{a(1-a)}}{2^t} + \frac{\pi^2}{2^{2t}}$$
using $2^t$ applications of $A$. For Grover ($a = M/N$) this gives $\tilde{M} = N\sin^2(\pi\tilde{\varphi})$ from the QPE estimate $\tilde{\varphi} = \theta/\pi$.

In [ ]:
# ── Estimate M by running QPE on the Grover operator ─────────────────────────
# Genuine quantum counting: run QPE (no shortcut to the exact eigenphase),
# read the peak clock outcome, and convert it to an M estimate.
n_ae      = 4
marked_ae = [1, 5, 9, 13]          # M=4 marked elements
M_ae      = len(marked_ae)
N_ae      = 2**n_ae
a_true    = M_ae / N_ae
theta_ae  = np.arcsin(np.sqrt(a_true))

O_f_ae = make_oracle(n_ae, marked_ae)

D_ae, s_ae = diffusion(n_ae)
G_ae = D_ae * O_f_ae

# Eigenstate of G with eigenvalue e^{+2iθ}: (|α⟩ - i|β⟩)/√2
alpha_ae = qt.Qobj(np.array([1/np.sqrt(N_ae-M_ae) if x not in marked_ae else 0.0
                               for x in range(N_ae)]), dims=[[2]*n_ae,[1]*n_ae])
beta_ae  = qt.Qobj(np.array([1/np.sqrt(M_ae) if x in marked_ae else 0.0
                               for x in range(N_ae)]), dims=[[2]*n_ae,[1]*n_ae])
eigenstate_G = (alpha_ae - 1j*beta_ae).unit()

# Run QPE for increasing clock precision and read off the peak each time
t_vals = range(3, 8)
M_estimates = []
for t_ae in t_vals:
    probs   = measure_clock(qpe(G_ae, eigenstate_G, t_ae), t_ae, n_ae)
    k_peak  = int(np.argmax(probs))
    phi_est = k_peak / 2**t_ae          # φ = θ/π
    M_estimates.append(N_ae * np.sin(np.pi * phi_est)**2)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(list(t_vals), M_estimates, 'o-', label='M estimate (from QPE peak)')
ax.axhline(M_ae, color='red', linestyle='--', label=f'true M={M_ae}')
ax.set_xlabel('clock qubits t'); ax.set_ylabel('estimated M')
ax.set_title('Amplitude estimation: QPE-based M estimate vs clock precision')
ax.legend(); plt.tight_layout(); plt.show()

print(f"True: M={M_ae}, a={a_true:.4f}, θ={np.degrees(theta_ae):.2f}°")
print(f"M estimates per t: {[round(m,3) for m in M_estimates]}")
print("Estimate converges to M=4 as the clock resolution increases.")

### Exercise 5.2 — Optimal step count

Take $n = 4$ qubits with `marked = [0, 5, 11]` ($M=3$).

**(a)** Compute $\theta = \arcsin(\sqrt{M/N})$ and the optimal step count $k^* = \lfloor\pi/(4\theta)\rfloor$.

**(b)** Run Grover for $k = 0, \ldots, 2k^*$ steps, recording $P_k = \|\Pi_\text{marked}|\psi_k\rangle\|^2$ at each step. Plot $P_k$ vs $k$ and mark $k^*$.

In [ ]:
# YOUR CODE HERE

n_ex2      = 4
marked_ex2 = [0, 5, 11]   # M=3
N_ex2      = 2**n_ex2

# (a) theta, k_star
# theta  = ...
# k_star = ...

# (b) iterate G, record P_k = (Pi_marked * psi).norm()**2, plot

In [ ]:
#@title 🔒 Exercise 5.2 (locked)  {display-mode: "form"}
reveal_solution("5.2", key="PASTE-KEY-HERE")


---
## Summary

| Topic | Key result |
|-------|-----------|
| Grover oracle | $O_f|x\rangle = (-1)^{f(x)}|x\rangle$; diffusion $D = 2|s\rangle\langle s| - \mathbb{I}$ |
| Grover iteration | $G = D\cdot O_f$ rotates by $2\theta$ in $\{|\alpha\rangle,|\beta\rangle\}$; optimal after $k^* = \lfloor\pi/(4\theta)\rfloor$ steps |
| Quadratic speedup | $k^* \approx \frac{\pi}{4}\sqrt{N/M}$ queries; $O(\sqrt{N})$ vs $O(N)$ classical unstructured search |
| Amplitude estimation | QPE on $G$; $\tilde{a} = \sin^2(\pi\tilde\varphi/2^t)$; $O(1/M)$ error — quadratic improvement over Monte Carlo |

**Next:** Notebook 6 covers Hamiltonian simulation via product formulas (§6).